In [7]:
import numpy as np
import pandas as pd

In [8]:
def backtest_long_flat(close, p_up, threshold=None, thr_series=None,
                       enter_thr=None, exit_thr=None, gate=None, slippage_bps=1.0):

    df = pd.concat({"close": close, "p_up": p_up}, axis=1).dropna()
    if gate is not None:
        g = gate.reindex(df.index).fillna(0).astype(bool)
    else:
        g = pd.Series(True, index=df.index)

    if thr_series is None:
        if threshold is None:
            threshold = 0.55
        thr_series = pd.Series(float(threshold), index=df.index)
    thr_series = thr_series.reindex(df.index)

    # Hysteresis defaults
    if enter_thr is None:
        enter_thr = thr_series
    if exit_thr is None:
        # a small cushion below enter threshold
        exit_thr = (thr_series - 0.01).clip(lower=0.0, upper=1.0)

    # Build position with hysteresis
    pos = pd.Series(0, index=df.index, dtype=int)
    holding = False
    for t in df.index:
        if not holding and g.loc[t] and (df.loc[t, "p_up"] >= enter_thr.loc[t]):
            holding = True
        elif holding and ((not g.loc[t]) or (df.loc[t, "p_up"] < exit_thr.loc[t])):
            holding = False
        pos.loc[t] = 1 if holding else 0

    # Trade next bar
    pos = pos.shift(1).fillna(0)
    ret = close.reindex(df.index).pct_change().fillna(0)
    strat = pos * ret

    delta_pos = pos.diff().fillna(0)
    costs = delta_pos.abs() * (slippage_bps / 10000.0)
    strat = strat - costs

    out = pd.DataFrame({"strat_ret": strat, "pos": pos, "delta_pos": delta_pos}, index=df.index)
    return out


In [9]:
def make_trend_gate(
    close: pd.Series,
    lookback: int = 120,
    slope_window: int = 10,
    min_slope: float = -0.0005,  # <- allow mildly negative slope
) -> pd.Series:
    sma   = close.rolling(lookback).mean()
    slope = sma.pct_change(slope_window)  # approx SMA slope (% change)
    gate  = (close > sma) & (slope > min_slope)
    return gate.fillna(False).astype(int)